# Create NER evaluation dataset

In [ ]:
# %load_ext autoreload
# %autoreload 2

## Init

In [ ]:
import logging
import sys
from logging.handlers import RotatingFileHandler
from pathlib import Path
from typing import Optional
import ast
import itertools
# Third-party imports (grouped for clarity)
# Note: Ensure all these are actually used in the script to avoid overhead
import base64
import difflib
import io
import json
import pprint
import random
import traceback
import urllib.request
from collections import Counter
from datetime import datetime

import argilla as rg
import html2text
import itables
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import regex as re
import requests
import seaborn as sns
from bs4 import BeautifulSoup
from dotenv import dotenv_values, find_dotenv
from huggingface_hub import DatasetCard, DatasetCardData
from tabulate import tabulate
from tqdm import tqdm
import spacy
from pathlib import Path
from wtpsplit import WtP, SaT

from typing import List, Union
from dataclasses import dataclass

# HuggingFace Datasets
from datasets import (Dataset, DatasetDict, DatasetInfo, Features, Sequence, Value, load_from_disk)

# Initialize Logger
import logging
from logging.config import dictConfig 
logger = logging.getLogger(__name__)
from archaeo_ner_greek.logging_config import LOGGING_CONFIG
dictConfig(LOGGING_CONFIG) 

# --- Configuration ---
PROJECT_NAME = "archaeo-ner-greek"
# Uses the user's home directory dynamically
BASE_DIR = Path.home() / "src" / "ekpa_2026" / PROJECT_NAME
DATA_DIR = BASE_DIR / "data"
LOG_DIR = Path.home() / "logs"

# Plotting setup
sns.set_theme()
itables.init_notebook_mode(all_interactive=False)

# --- Initialization ---
# Load environment variables
env_path = find_dotenv()
env_vars = dotenv_values()

# Create directories
DATA_DIR.mkdir(parents=True, exist_ok=True)

from archaeo_ner_greek.utils import configure_argilla_client, configure_argilla_resources

logger.info(f"Finished initializing environment.")
logger.debug(f"Using {env_path}.")
logger.debug(f"Data directory set to: {DATA_DIR}")
logger.info(f"Environment variables: {env_vars}")

### Configure the evaluation dataset



In [ ]:
def setup_argilla_dataset(client: rg.Argilla, dataset_name: str, workspace_name: str, guidelines: str):
    """
    Recreates the Argilla dataset schema.
    """
    try:
        # Check and delete existing dataset
        for cdataset in client.datasets:
            if cdataset.name == dataset_name:
                logger.info(f"Deleting existing dataset: {dataset_name}")
                cdataset.delete()
        
        # Define Settings
        settings = rg.Settings(
            guidelines=guidelines,
            fields=[
                rg.TextField(name="prev_sentences_field", title="Previous sentences", required=True),
                rg.TextField(name="sentence_field", title="Sentence", required=True),
                rg.TextField(name="next_sentences_field", title="Next sentences", required=True),
                rg.TextField(name="document_sentence_id_field", title="Document Sentence ID", required=True),
            ],
            metadata=[
                rg.IntegerMetadataProperty(name="sentence_id_metadata", title="Sentence ID", visible_for_annotators=True),
            ],
            questions=[
                rg.SpanQuestion(
                    name="entities",
                    title="Entities",
                    field="sentence_field",
                    labels=['ARTEFACT', 'PERIOD', "LOCATION", 'CONTEXT', 'MATERIAL', 'SPECIES', "FEATURE", "PERSON", "MISC"],
                    allow_overlapping=True,
                ),
                rg.TextQuestion(
                    name="label_suggestion",
                    title="Label suggestion",
                    description="Suggest a new label for MISC items.",
                ),
            ],
        )
        
        # Create Dataset
        dataset = rg.Dataset(
            name=dataset_name,
            workspace=workspace_name, 
            settings=settings,
            client=client,
        )
        dataset.create()
        logger.info(f"Dataset {dataset_name} created successfully.")
        return dataset

    except Exception:
        logger.error("Failed to setup Argilla dataset", exc_info=True)
        return None

### Add records

In [ ]:
# Optimization: Compile regex globally once
LIST_ITEM_PATTERN = re.compile(r"^\s*\d+[\.\)-]\s*$")

def merge_list_fragments(sentences: List[str]) -> List[str]:
    """
    Merges sentences that are fragments of a numbered list using the pre-compiled pattern.
    """
    merged_sentences = []
    
    for sentence in sentences:
        if not merged_sentences:
            merged_sentences.append(sentence)
            continue
            
        previous_sentence = merged_sentences[-1]
        
        if LIST_ITEM_PATTERN.match(previous_sentence):
            merged_sentences[-1] = f"{previous_sentence}{sentence}"
            logger.debug(f"Merged '{sentence}' into '{previous_sentence}'")
        else:
            merged_sentences.append(sentence)
            
    return merged_sentences

def create_annotation_dataframe(
    file_paths: List[Union[str, Path]], 
    context_window_size: int = 2
) -> pd.DataFrame:
    """
    Processes text files into a DataFrame for sentence-level annotation.
    Optimized to use nlp.pipe for faster tokenization.
    """
    
    # Load Models
    try:
        # Disable unnecessary pipeline components for speed if only tokenization is needed
        nlp = spacy.blank("el") 
        # Consider switching to 'SaT' if performance is still an issue
        sat = SaT("sat-3l", style_or_domain="ud", language="el")
    except Exception:
        logger.error("Failed to load models.")
        raise

    data_rows = []

    for file_path in file_paths:
        path = Path(file_path)
        doc_id = path.stem
        
        logger.info(f"Processing document: {doc_id}")
        
        try:
            raw_text = path.read_text(encoding="utf-8", errors="replace")
            raw_lines = [line for line in raw_text.splitlines() if line.strip()]
            
            if not raw_lines:
                logger.warning(f"File {doc_id} is empty or contains only whitespace.")
                continue

            # Batch Split via SaT
            batch_results = sat.split(raw_lines)
            
            sentences = list(itertools.chain.from_iterable(batch_results))
            sentences = merge_list_fragments(sentences)
            
            total_sentences = len(sentences)

            # Optimization: Use nlp.pipe for batch processing of all sentences in the document
            # n_process can be increased if you have multiple CPU cores available
            sentence_docs = list(nlp.pipe(sentences))

            for i, (sentence_text, doc) in enumerate(zip(sentences, sentence_docs)):
                
                row_id = f"{doc_id}_{i}"
                
                # Context generation logic
                if context_window_size == -1:
                    prev_slice = sentences[:i]
                    next_slice = sentences[i+1:]
                else:
                    start_prev = max(0, i - context_window_size)
                    end_next = min(total_sentences, i + 1 + context_window_size)
                    prev_slice = sentences[start_prev:i]
                    next_slice = sentences[i+1:end_next]
                
                prev_text = "\n".join(prev_slice)
                next_text = "\n".join(next_slice)

                # Extract tokens with offsets from the pre-processed doc
                tokens_with_offsets = [
                    {
                        "token": token.text,
                        "start": token.idx,
                        "end": token.idx + len(token.text)
                    }
                    for token in doc
                ]

                data_rows.append({
                    "id": row_id,
                    "text": tokens_with_offsets,
                    "prev_sentences": prev_text,
                    "next_sentences": next_text
                })
                
        except Exception:
            logger.error(f"Error processing file {path}", exc_info=True)
            continue

    return pd.DataFrame(data_rows)




In [ ]:
def log_annotation_records(df, dataset):
    """
    Processes annotation records and logs them to Argilla.
    
    Args:
        df (pd.DataFrame): The dataframe containing annotations.
        dataset (rg.Dataset): The Argilla dataset object.
    """
    # --- OPTIMIZATION START ---
    # 1. Pre-fill missing values on the whole column at once (Much faster than per row)
    # Using '---' as default for both NaN and empty strings
    df['prev_sentences'] = df['prev_sentences'].fillna("---").replace("", "---")
    df['next_sentences'] = df['next_sentences'].fillna("---").replace("", "---")

    # 2. Pre-process the text tokens (Vectorized-ish approach)
    # We construct the sentence string once here so we don't do it inside the object creation loop
    # Assuming 'text' is a list of dicts: [{'token': 'Hello'}, {'token': 'World'}]
    texts = df["text"].apply(lambda x: ' '.join([t.get("token", "") for t in x])).tolist()
    
    # --- RECORD CREATION ---
    records = []
    
    # 3. Use zip() instead of iterrows(). 
    # iterrows is very slow because it creates a Series for every row. 
    # zip iterates over native numpy arrays/lists which is instant.
    print("Building records...")
    
    # We zip the columns we need. 
    # Note: df.index is used for 'idx', and the rest match your fields.
    iterator = zip(
        df.index, 
        texts, 
        df['prev_sentences'], 
        df['next_sentences'], 
        df['id']
    )

    for idx, text_val, prev, next_val, doc_id in tqdm(iterator, total=len(df)):
        try:
            record = rg.Record(
                fields={
                    "prev_sentences_field": prev,
                    "sentence_field": text_val,
                    "next_sentences_field": next_val,
                    "document_sentence_id_field": doc_id,
                },
                metadata={
                    "sentence_id_metadata": idx,
                },
                # Uncomment suggestions if needed, passing data via zip above
            )
            records.append(record)
        except Exception:
            # It is better to print the specific error for the specific index
            print(f"Error processing index {idx}")
            traceback.print_exc()

    # --- LOGGING ---
    if records:
        logger.info(f"Number of records to log: {len(records)}")
        
        # Delete pending records if necessary
        try:
            status_filter = rg.Query(filter=rg.Filter(("response.status", "==", "pending")))
            # Note: Fetching records can be slow. Only do this if you actually intend to delete.
            records_to_delete = list(dataset.records(status_filter)) 
            dataset.records.delete(records_to_delete) 
        except Exception:
            logger.warning("Failed to filter/delete old records.")
            pass

        # Log new records
        try:
            # batch_size is automatically handled by Argilla, but good to know it exists
            dataset.records.log(records)
            logger.info("Records logged successfully.")
        except Exception:
            logger.error("Failed to log records to Argilla.")
            traceback.print_exc()


In [ ]:
@dataclass
class PipelineConfig:
    recreate_workspaces: bool
    recreate_schema: bool
    reprocess_text: bool
    upload_records: bool
    cache_path: Path

def run_pipeline(cfg: PipelineConfig, env_vars: dict):
    logger.info("Starting pipeline execution...")
    
    # --- Step 1: Environment & Client Setup ---
    # Initialize the client using the provided utility function
    try:
        client = configure_argilla_client(env_vars=env_vars)
    except Exception as e:
        logger.error("Failed to initialize Argilla client.", exc_info=True)
        return

    # Conditionally recreate workspaces and users
    if cfg.recreate_workspaces:
        logger.info("Recreating workspaces and users...")
        configure_argilla_resources(client, env_vars)
    else:
        logger.info("Skipping workspace/user recreation.")

    # --- Step 2: Dataset Schema Management ---
    # Pass the initialized client to the setup function
    workspace_name = ast.literal_eval(env_vars["ARGILLA_WORKSPACES"])[0]["name"]
    dataset_name = env_vars["ARGILLA_DATASET"]

    with open(DATA_DIR / "archaeobert_ner_gudelines_mt_translation.md") as inf:
        guidelines = inf.read()  

    dataset = None
    if cfg.recreate_schema:
        # Assuming setup_argilla_dataset is defined as in the previous suggestion
        dataset = setup_argilla_dataset(client, dataset_name, workspace_name, guidelines)
    else:
        try:
            dataset = client.datasets(name=dataset_name, workspace=workspace_name)
            logger.info(f"Using existing dataset: {dataset_name}")
        except Exception:
            logger.warning(f"Dataset {dataset_name} not found. Consider setting recreate_schema=True.")

    # --- Step 3: Data Processing or Loading ---
    df_annotation = None
    
    if cfg.reprocess_text:
        logger.info("Reprocessing source text files...")
        files = list(DATA_DIR.glob("*.txt"))
        # Assuming create_annotation_dataframe is defined
        df_annotation = create_annotation_dataframe(files, context_window_size=3)
        
        # Cache the results
        df_annotation.to_pickle(cfg.cache_path)
        logger.info(f"Data cached to {cfg.cache_path}")
        
    else:
        if cfg.cache_path.exists():
            logger.info(f"Loading data from cache: {cfg.cache_path}")
            df_annotation = pd.read_pickle(cfg.cache_path)
        else:
            logger.error(f"Cache file not found at {cfg.cache_path}. Cannot proceed without reprocessing.")
            return

    # --- Step 4: Record Upload ---
    if cfg.upload_records and df_annotation is not None and dataset is not None:
        logger.info("Starting record upload...")
        # Assuming log_annotation_records is defined
        log_annotation_records(df_annotation, dataset)
    elif cfg.upload_records:
        logger.warning("Skipping upload: DataFrame or Dataset is missing.")

# --- CONTROL PANEL ---
config = PipelineConfig(
    recreate_workspaces=False,   # Set to True to reset users/workspaces
    recreate_schema=False,        # Set to True to rebuild dataset
    reprocess_text=False,        # Set to True to parse text files again
    upload_records=False,         # Set to True to push data
    cache_path=DATA_DIR / "processed_dataframe_cache.pkl"
)

# Execute the pipeline passing the environment variables
run_pipeline(config, env_vars)

In [ ]:
client = configure_argilla_client(env_vars= env_vars)

In [ ]:
workspace = client.workspaces("archaeo_ner_greek")
print(workspace)